# Experiment 1: Linear Mixed Models

## Theoretical Framework

**Core effects** (can exist as main effects):
- **CT**: Central Tendency (curDur) - direct effect on bias
- **SD**: Serial Dependence (preDur) - direct effect on bias

**Modulators** (only exist as interactions):
- **curCoh**: Current uncertainty - modulates SD strength
- **preCoh**: Previous uncertainty - modulates SD strength
- **SameSwitch**: Gating factor - modulates prior integration

## Experiment 1 Hypothesis (Weighting)

**Question**: Does perceptual uncertainty (curCoh) modulate serial dependence strength?

**Key interaction**: `preDur × curCoh`

**Prediction**: When current input is unreliable (low curCoh), prior information (preDur) is weighted more heavily → stronger SD.

## Variable Coding
- Duration: centered at 1.2s
- Coherence: original values (0.3, 0.7)
- SameSwitch: Same=0, Switch=1

In [1]:
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.formula.api as smf
import warnings
from IPython.display import display
warnings.filterwarnings('ignore')
%config InlineBackend.figure_format = 'retina'

## 1. Data Preparation

In [2]:
def prepare_lmm_data(df):
    """
    Prepare variables for LMM analysis.
    
    Coding:
    - Duration: centered at 1.2s
    - Coherence: original values (0.3, 0.7) - NOT centered
    - SameSwitch: Same=0, Switch=1
    """
    df = df.copy()
    
    # Remove outliers
    df = df[df['is_outlier'] == False].copy()
    
    # Center duration variables
    df['curDur'] = df['curDur'] - 1.2
    df['preDur'] = df['preDur1back'] - 1.2
    
    # Coherence: original values (NOT centered)
    df['curCoh'] = df['curCoherence']
    df['preCoh'] = df['preCoherence1back']
    
    # SameSwitch: Same=0, Switch=1
    df['SameSwitch'] = (df['preCoherence1back'] != df['curCoherence']).astype(int)
    
    # TransitionType for appendix models
    df['TransitionType'] = df['TransitionType']
    
    # Drop missing values
    df = df.dropna(subset=['preDur', 'preCoherence1back'])
    
    # Summary
    print(f"Data prepared: {len(df)} trials, {df['subID'].nunique()} subjects")
    print(f"\nVariables:")
    print(f"  curDur: [{df['curDur'].min():.2f}, {df['curDur'].max():.2f}] (centered)")
    print(f"  preDur: [{df['preDur'].min():.2f}, {df['preDur'].max():.2f}] (centered)")
    print(f"  curCoh: {sorted(df['curCoh'].unique())}")
    print(f"  preCoh: {sorted(df['preCoh'].unique())}")
    print(f"  SameSwitch: {df['SameSwitch'].value_counts().to_dict()}")
    
    return df

In [3]:
# Load and prepare data
df_raw = pd.read_pickle('../../data/experiment1/E1.pkl')
df_lmm = prepare_lmm_data(df_raw)

Data prepared: 4677 trials, 22 subjects

Variables:
  curDur: [-0.40, 0.40] (centered)
  preDur: [-0.40, 0.40] (centered)
  curCoh: [np.float64(0.3), np.float64(0.7)]
  preCoh: [np.float64(0.3), np.float64(0.7)]
  SameSwitch: {1: 2409, 0: 2268}


## 2. Model Definitions

### Theoretical Structure

| Variable | Role | Rationale |
|----------|------|-----------|
| curDur | Main effect | CT directly biases reproduction |
| preDur | Main effect | SD directly biases reproduction |
| curCoh | Modulator only | Uncertainty modulates SD strength |
| preCoh | Modulator only | Previous uncertainty modulates SD |
| SameSwitch | Interaction only | Gating modulates prior integration |

### Model Hierarchy

```
Level 0: Null
Level 1: Core (CT + SD)
Level 2: + Weighting interaction (preDur × curCoh) ← Manuscript LMM-E2
Level 3: + Gating interaction (preDur × SameSwitch)
Level 4: Appendix models (TransitionType, three-way)
```

In [4]:
def get_model_formulas():
    """
    Experiment 1 model hierarchy.
    
    Theoretical constraints:
    - curCoh, preCoh: modulators only (no main effects)
    - SameSwitch: gating factor (interaction only)
    """
    formulas = {
        # ===== Level 0: Null =====
        'M0_Null': 'curBias ~ 1',
        
        # ===== Level 1: Core Effects =====
        'M1_CT': 'curBias ~ curDur',
        'M2_SD': 'curBias ~ preDur',
        'M3_Core': 'curBias ~ curDur + preDur',
        
        # ===== Level 2: Weighting Hypothesis =====
        # preDur × curCoh: SD modulated by current uncertainty
        'M4_Weighting': 'curBias ~ curDur + preDur + preDur:curCoh',
        
        # ===== Level 3: Previous Uncertainty =====
        # preDur × preCoh: SD modulated by previous uncertainty
        'M5_PreUncert': 'curBias ~ curDur + preDur + preDur:preCoh',
        
        # ===== Level 4: Gating =====
        # preDur × SameSwitch: SD modulated by gating
        'M6_Gating': 'curBias ~ curDur + preDur + preDur:SameSwitch',
        
        # ===== Level 5: Combined =====
        'M7_Weight_Gate': 'curBias ~ curDur + preDur + preDur:curCoh + preDur:SameSwitch',
        
        # ===== Level 6: Full Model (with curCoh main effect) =====
        'M8_Full': 'curBias ~ curDur + preDur + curCoh + preDur:curCoh',
        
        # ===== Appendix: Alternative Model 1 (TransitionType) =====
        'Alt1_TT': 'curBias ~ curDur + preDur * C(TransitionType)',
        
        # ===== Appendix: Alternative Model 2 (Three-way) =====
        'Alt2_3way': 'curBias ~ preDur * preCoh * SameSwitch',
    }
    return formulas

## 3. Model Fitting Functions

In [5]:
def fit_lmm(df, formulas=None, method='lbfgs', reml=True, verbose=True):
    """
    Fit all hierarchical models using REML.
    """
    if formulas is None:
        formulas = get_model_formulas()
    
    results = {}
    for name, formula in formulas.items():
        try:
            model = smf.mixedlm(formula, df, groups=df['subID'])
            result = model.fit(method=method, reml=reml, disp=False)
            
            # Check for convergence issues
            if np.isinf(result.llf) or np.isnan(result.llf):
                if verbose:
                    print(f"  [WARN] {name}: LogLik is inf/nan, trying alternative methods...")
                for alt_method in ['powell', 'cg', 'nm']:
                    try:
                        result = model.fit(method=alt_method, reml=reml, disp=False)
                        if not np.isinf(result.llf) and not np.isnan(result.llf):
                            if verbose:
                                print(f"       -> Succeeded with {alt_method}")
                            break
                    except:
                        continue
            
            results[name] = result
            if verbose:
                status = "OK" if not np.isinf(result.llf) else "WARN (inf)"
                print(f"  [{status}] {name}")
        except Exception as e:
            if verbose:
                print(f"  [FAIL] {name}: {e}")
            results[name] = None
    
    return results


def compute_model_fit(result, n_obs):
    """
    Compute AIC and BIC manually.
    """
    if result is None:
        return np.nan, np.nan
    
    k = len(result.params)
    llf = result.llf
    
    if np.isinf(llf) or np.isnan(llf):
        return np.nan, np.nan
    
    aic = 2 * k - 2 * llf
    bic = k * np.log(n_obs) - 2 * llf
    
    return aic, bic


def compare_models(results, n_obs):
    """Create model comparison table."""
    comparison = []
    
    for name, res in results.items():
        if res is None:
            comparison.append({
                'Model': name,
                'nParams': np.nan,
                'LogLik': np.nan,
                'AIC': np.nan,
                'BIC': np.nan,
            })
            continue
        
        aic, bic = compute_model_fit(res, n_obs)
        
        comparison.append({
            'Model': name,
            'nParams': len(res.params),
            'LogLik': res.llf,
            'AIC': aic,
            'BIC': bic,
        })
    
    comp_df = pd.DataFrame(comparison)
    
    # Compute delta AIC/BIC
    valid_aic = comp_df['AIC'].dropna()
    valid_bic = comp_df['BIC'].dropna()
    
    if len(valid_aic) > 0:
        comp_df['dAIC'] = comp_df['AIC'] - valid_aic.min()
    else:
        comp_df['dAIC'] = np.nan
        
    if len(valid_bic) > 0:
        comp_df['dBIC'] = comp_df['BIC'] - valid_bic.min()
    else:
        comp_df['dBIC'] = np.nan
    
    return comp_df

## 4. Fit All Models

In [6]:
print("Fitting hierarchical models (REML estimation)...\n")
lmm_results = fit_lmm(df_lmm, reml=True)

Fitting hierarchical models (REML estimation)...

  [OK] M0_Null
  [OK] M1_CT
  [OK] M2_SD
  [OK] M3_Core
  [OK] M4_Weighting
  [OK] M5_PreUncert
  [OK] M6_Gating
  [OK] M7_Weight_Gate
  [OK] M8_Full
  [OK] Alt1_TT
  [OK] Alt2_3way


## 5. Model Comparison

In [7]:
def print_comparison_table(comp_df, title="MODEL COMPARISON"):
    """Print formatted comparison table."""
    print("=" * 100)
    print(title)
    print("=" * 100)
    print(f"{'Model':<20} {'k':>4} {'LogLik':>12} {'AIC':>12} {'dAIC':>8} {'BIC':>12} {'dBIC':>8}")
    print("-" * 100)
    
    for _, row in comp_df.iterrows():
        if pd.isna(row['LogLik']):
            print(f"{row['Model']:<20} {'--':>4} {'--':>12} {'--':>12} {'--':>8} {'--':>12} {'--':>8}")
        else:
            print(f"{row['Model']:<20} {int(row['nParams']):>4} {row['LogLik']:>12.2f} {row['AIC']:>12.2f} {row['dAIC']:>8.2f} {row['BIC']:>12.2f} {row['dBIC']:>8.2f}")
    
    print("=" * 100)
    
    # Find best models
    valid = comp_df.dropna(subset=['AIC'])
    if len(valid) > 0:
        best_aic = valid.loc[valid['AIC'].idxmin(), 'Model']
        best_bic = valid.loc[valid['BIC'].idxmin(), 'Model']
        print(f"\nBest by AIC: {best_aic}")
        print(f"Best by BIC: {best_bic}")

In [8]:
# Create comparison table
n_obs = len(df_lmm)
comp_df = compare_models(lmm_results, n_obs)
print_comparison_table(comp_df, title="MODEL COMPARISON - EXPERIMENT 1")

MODEL COMPARISON - EXPERIMENT 1
Model                   k       LogLik          AIC     dAIC          BIC     dBIC
----------------------------------------------------------------------------------------------------
M0_Null                 2       170.68      -337.36  1233.52      -324.46  1207.72
M1_CT                   3       691.95     -1377.90   192.98     -1358.55   173.62
M2_SD                   3       175.98      -345.97  1224.91      -326.61  1205.56
M3_Core                 4       695.95     -1383.90   186.98     -1358.09   174.08
M4_Weighting            5       696.52     -1383.04   187.83     -1350.79   181.38
M5_PreUncert            5       694.28     -1378.56   192.32     -1346.31   185.86
M6_Gating               5       693.04     -1376.08   194.80     -1343.83   188.35
M7_Weight_Gate          6       693.61     -1375.23   195.65     -1336.52   195.65
M8_Full                 6       791.44     -1570.88     0.00     -1532.17     0.00
Alt1_TT                10       777.2

In [9]:
# Sort by AIC and display
print("\nModels Ranked by AIC:")
comp_sorted = comp_df.dropna(subset=['AIC']).sort_values('AIC')
display(comp_sorted[['Model', 'nParams', 'LogLik', 'AIC', 'dAIC', 'BIC', 'dBIC']].round(2))


Models Ranked by AIC:


,Model,nParams,LogLik,AIC,dAIC,BIC,dBIC
8,M8_Full,6,791.44,-1570.88,0.00,-1532.17,0.00
9,Alt1_TT,10,777.20,-1534.40,36.47,-1469.90,62.27
3,M3_Core,4,695.95,-1383.90,186.98,-1358.09,174.08
4,M4_Weighting,5,696.52,-1383.04,187.83,-1350.79,181.38
5,M5_PreUncert,5,694.28,-1378.56,192.32,-1346.31,185.86
1,M1_CT,3,691.95,-1377.90,192.98,-1358.55,173.62
6,M6_Gating,5,693.04,-1376.08,194.80,-1343.83,188.35
7,M7_Weight_Gate,6,693.61,-1375.23,195.65,-1336.52,195.65
10,Alt2_3way,9,240.14,-462.29,1108.59,-404.24,1127.94
2,M2_SD,3,175.98,-345.97,1224.91,-326.61,1205.56


## 6. Best Model Details

In [10]:
def print_model_summary(result, model_name="Model"):
    """Print detailed model results."""
    if result is None:
        print(f"{model_name}: Model failed to fit")
        return
    
    print(f"\n{'=' * 90}")
    print(f"{model_name}")
    print(f"{'=' * 90}")
    print(f"{'Parameter':<40} {'Beta':>10} {'SE':>10} {'z':>10} {'p':>12}")
    print("-" * 90)
    
    for param in result.params.index:
        if param == 'Group Var':
            continue
        beta = result.params[param]
        se = result.bse.get(param, np.nan)
        z = result.tvalues.get(param, np.nan)
        p = result.pvalues.get(param, np.nan)
        
        if pd.isna(p):
            sig, p_str = '', 'NA'
        elif p < 0.001:
            sig, p_str = '***', '< .001'
        elif p < 0.01:
            sig, p_str = '**', f'{p:.4f}'
        elif p < 0.05:
            sig, p_str = '*', f'{p:.4f}'
        else:
            sig, p_str = '', f'{p:.4f}'
        
        print(f"{param:<40} {beta:>10.4f} {se:>10.4f} {z:>10.2f} {p_str:>10} {sig}")
    
    print("-" * 90)
    print(f"Random Intercept Var: {result.cov_re.iloc[0,0]:.4f}")
    print(f"N obs: {int(result.nobs)}, N groups: {len(result.random_effects)}")
    aic, bic = compute_model_fit(result, result.nobs)
    print(f"AIC: {aic:.2f}, BIC: {bic:.2f}")

In [11]:
# Print the theoretically-driven weighting model
print_model_summary(lmm_results['M4_Weighting'], "M4_Weighting (Theory): curDur + preDur + preDur:curCoh")


M4_Weighting (Theory): curDur + preDur + preDur:curCoh
Parameter                                      Beta         SE          z            p
------------------------------------------------------------------------------------------
Intercept                                    0.0915     0.0292       3.14     0.0017 **
curDur                                      -0.3698     0.0108     -34.25     < .001 ***
preDur                                       0.1025     0.0288       3.56     < .001 ***
preDur:curCoh                               -0.1210     0.0532      -2.28     0.0228 *
------------------------------------------------------------------------------------------
Random Intercept Var: 0.0185
N obs: 4677, N groups: 22
AIC: -1383.04, BIC: -1350.79


In [12]:
# Print full model (with curCoh main effect)
print_model_summary(lmm_results['M8_Full'], "M8_Full: curDur + preDur + curCoh + preDur:curCoh")


M8_Full: curDur + preDur + curCoh + preDur:curCoh
Parameter                                      Beta         SE          z            p
------------------------------------------------------------------------------------------
Intercept                                   -0.0129     0.0303      -0.43     0.6689 
curDur                                      -0.3710     0.0106     -35.08     < .001 ***
preDur                                       0.0995     0.0282       3.53     < .001 ***
curCoh                                       0.2088     0.0147      14.16     < .001 ***
preDur:curCoh                               -0.1164     0.0521      -2.24     0.0253 *
------------------------------------------------------------------------------------------
Random Intercept Var: 0.0188
N obs: 4677, N groups: 22
AIC: -1570.88, BIC: -1532.17


In [13]:
# Print the actual best model by AIC
valid = comp_df.dropna(subset=['AIC'])
if len(valid) > 0:
    best_aic_name = valid.loc[valid['AIC'].idxmin(), 'Model']
    best_bic_name = valid.loc[valid['BIC'].idxmin(), 'Model']
    
    print_model_summary(lmm_results[best_aic_name], f"{best_aic_name} (Best by AIC)")
    
    if best_bic_name != best_aic_name:
        print_model_summary(lmm_results[best_bic_name], f"{best_bic_name} (Best by BIC)")


M8_Full (Best by AIC)
Parameter                                      Beta         SE          z            p
------------------------------------------------------------------------------------------
Intercept                                   -0.0129     0.0303      -0.43     0.6689 
curDur                                      -0.3710     0.0106     -35.08     < .001 ***
preDur                                       0.0995     0.0282       3.53     < .001 ***
curCoh                                       0.2088     0.0147      14.16     < .001 ***
preDur:curCoh                               -0.1164     0.0521      -2.24     0.0253 *
------------------------------------------------------------------------------------------
Random Intercept Var: 0.0188
N obs: 4677, N groups: 22
AIC: -1570.88, BIC: -1532.17


## 7. Summary for Manuscript

In [14]:
# Model comparison
print("\n" + "=" * 80)
print("MODEL COMPARISON - EXPERIMENT 1")
print("=" * 80)

key_models = ['M3_Core', 'M4_Weighting', 'M5_PreUncert', 'M6_Gating', 'M7_Weight_Gate', 'M8_Full']
key_comp = comp_df[comp_df['Model'].isin(key_models)].sort_values('AIC')

print("\nKey Models (sorted by AIC):")
display(key_comp[['Model', 'nParams', 'LogLik', 'AIC', 'dAIC', 'BIC', 'dBIC']].round(2))


MODEL COMPARISON - EXPERIMENT 1

Key Models (sorted by AIC):


,Model,nParams,LogLik,AIC,dAIC,BIC,dBIC
8,M8_Full,6,791.44,-1570.88,0.00,-1532.17,0.00
3,M3_Core,4,695.95,-1383.90,186.98,-1358.09,174.08
4,M4_Weighting,5,696.52,-1383.04,187.83,-1350.79,181.38
5,M5_PreUncert,5,694.28,-1378.56,192.32,-1346.31,185.86
6,M6_Gating,5,693.04,-1376.08,194.80,-1343.83,188.35
7,M7_Weight_Gate,6,693.61,-1375.23,195.65,-1336.52,195.65


In [15]:
# Key findings
print("\n" + "=" * 80)
print("KEY FINDINGS - EXPERIMENT 1")
print("=" * 80)

# Weighting interaction from M4_Weighting
res_weight = lmm_results['M4_Weighting']
print("\n1. Weighting Model (M4_Weighting):")
b = res_weight.params.get('preDur:curCoh', np.nan)
p = res_weight.pvalues.get('preDur:curCoh', np.nan)
sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''
print(f"   preDur:curCoh: β = {b:.4f}, p = {p:.4f} {sig}")

# Full model
res_full = lmm_results['M8_Full']
print("\n2. Full Model (M8_Full):")
b = res_full.params.get('preDur:curCoh', np.nan)
p = res_full.pvalues.get('preDur:curCoh', np.nan)
sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''
print(f"   preDur:curCoh: β = {b:.4f}, p = {p:.4f} {sig}")

# Compare AIC
aic_weight, _ = compute_model_fit(res_weight, n_obs)
aic_full, _ = compute_model_fit(res_full, n_obs)
print(f"\n3. Model Fit Comparison:")
print(f"   M4_Weighting (parsimonious): AIC = {aic_weight:.2f}")
print(f"   M8_Full (with curCoh main effect): AIC = {aic_full:.2f}")

# Interpretation
print("\n4. Interpretation:")
b_weight = res_weight.params.get('preDur:curCoh', 0)
if b_weight < 0:
    print("   Negative preDur:curCoh → Lower curCoh (more uncertainty) → Stronger SD")
    print("   Supports Bayesian weighting hypothesis.")
else:
    print("   Positive preDur:curCoh → Higher curCoh → Stronger SD")


KEY FINDINGS - EXPERIMENT 1

1. Weighting Model (M4_Weighting):
   preDur:curCoh: β = -0.1210, p = 0.0228 *

2. Full Model (M8_Full):
   preDur:curCoh: β = -0.1164, p = 0.0253 *

3. Model Fit Comparison:
   M4_Weighting (parsimonious): AIC = -1383.04
   M8_Full (with curCoh main effect): AIC = -1570.88

4. Interpretation:
   Negative preDur:curCoh → Lower curCoh (more uncertainty) → Stronger SD
   Supports Bayesian weighting hypothesis.


In [16]:
# Save results
comp_df.to_csv('lmm_model_comparison.csv', index=False)
print("Model comparison saved to lmm_model_comparison.csv")

Model comparison saved to lmm_model_comparison.csv


## 8. Cross-Experiment Summary

Compare the key findings between experiments.

In [17]:
print("\n" + "=" * 70)
print("EXPERIMENT 1 SUMMARY")
print("=" * 70)
print("\nExpected: T2_Weighting (preDur x curCoh) should be preferred")
print("         because ramped coherence reduces category salience.")
print("\nKey question: Does uncertainty modulate serial dependence strength?")


EXPERIMENT 1 SUMMARY

Expected: T2_Weighting (preDur x curCoh) should be preferred
         because ramped coherence reduces category salience.

Key question: Does uncertainty modulate serial dependence strength?
